In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
url = "https://books.toscrape.com/catalogue/page-1.html"

In [ ]:
page_content = requests.get(url).text

In [ ]:
website = BeautifulSoup(page_content, "html.parser")

In [ ]:
website.title

In [ ]:
books = website.find_all("article", class_="product_pod")

In [ ]:
len(books)

In [ ]:
# Test: print book names from page 1
for b in books:
    print(b.find("h3").find("a")["title"])

In [ ]:
# Test: print prices from page 1
for b in books:
    print(b.find("p", class_="price_color").text.strip())

In [ ]:
# Test: print stock status from page 1
for b in books:
    print(b.find("p", class_="instock").text.strip())

In [ ]:
# Function to get description from a book's individual page
def get_description(book_url):
    r = requests.get(book_url)
    soup = BeautifulSoup(r.text, "html.parser")
    meta = soup.find("meta", attrs={"name": "description"})
    if meta:
        return meta["content"].strip()
    return "No description"

In [ ]:
# Test description on one book
first_book = books[0]
relative_url = first_book.find("h3").find("a")["href"]
book_url = "https://books.toscrape.com/catalogue/" + relative_url
print(get_description(book_url))

In [ ]:
# Scrape all 50 pages
BASE_URL = "https://books.toscrape.com/catalogue/"
PAGE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

results = []

for page_num in range(1, 51):
    print(page_num, PAGE_URL.format(page_num))
    
    r = requests.get(PAGE_URL.format(page_num))
    soup = BeautifulSoup(r.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")
    
    for b in books:
        name  = b.find("h3").find("a")["title"]
        price = b.find("p", class_="price_color").text.strip()
        stock = b.find("p", class_="instock").text.strip()
        
        relative_url = b.find("h3").find("a")["href"]
        book_url = BASE_URL + relative_url
        desc = get_description(book_url)
        
        results.append({
            "book name": name,
            "book description": desc,
            "book price": price,
            "stock": stock,
            "page": page_num
        })

In [ ]:
df = pd.DataFrame(results)
df

In [ ]:
df.to_csv("books_data.csv", index=False)